In [0]:
#/Volumes/workspace/default/pysparktutorial_al_dataset_bigmartsales/BigMart Sales.csv

#### Read JSON

In [0]:
df_json = spark.read.format('json')\
            .option('inferschema', True)\
            .option('header', True)\
            .option('multiline', False)\
            .load('/Volumes/workspace/default/pysparktutorial_al_dataset_bigmartsales/drivers.json')

In [0]:
df_json.display()

In [0]:
dbutils.fs.ls("/Volumes/workspace/default/pysparktutorial_al_dataset_bigmartsales")

In [0]:
df = spark.read.format('csv').option('inferSchema',True).option('header',True).load('/Volumes/workspace/default/pysparktutorial_al_dataset_bigmartsales/BigMart Sales.csv')

In [0]:
df.show()

In [0]:
df.display()

## Day 2 - Define Schema of Dataframe

### Schema Definition

In [0]:
df.printSchema()

### DDL Schema

- Update Item_Weight datatype from double to string

In [0]:
my_ddl_schema = '''
                    Item_Identifier STRING,
                    Item_Weight STRING,
                    Item_Fat_Content STRING, 
                    Item_Visibility DOUBLE,
                    Item_Type STRING,
                    Item_MRP DOUBLE,
                    Outlet_Identifier STRING,
                    Outlet_Establishment_Year INT,
                    Outlet_Size STRING,
                    Outlet_Location_Type STRING, 
                    Outlet_Type STRING,
                    Item_Outlet_Sales DOUBLE 

                ''' 

In [0]:
# df = spark.read.format('csv')\
#         .schema(my_ddl_schema)\
#         .option('header', True)\
#         .load('/Volumes/workspace/default/pysparktutorial_al_dataset_bigmartsales/BigMart Sales.csv')

In [0]:
df.display()

In [0]:
df.printSchema()

### StructType Schema
- Update all column to String data type.

In [0]:
from pyspark.sql.types import * 

In [0]:
# my_strct_schema = StructType(
#                                 [ 
#                                     StructField('Item_Identifier',StringType(),True),
#                                     StructField('Item_Weight',StringType(),True),
#                                     StructField('Item_Fat_Content',StringType(),True), 
#                                     StructField('Item_Visibility',StringType(),True), 
#                                     StructField('Item_MRP',StringType(),True),
#                                     StructField('Outlet_Identifier',StringType(),True),
#                                     StructField('Outlet_Establishment_Year',StringType(),True),
#                                     StructField('Outlet_Size',StringType(),True), 
#                                     StructField('Outlet_Location_Type',StringType(),True),
#                                     StructField('Outlet_Type',StringType(),True),
#                                     StructField('Item_Outlet_Sales',StringType(),True)

#                                 ]
#                             )

# df = spark.read.format('csv')\
#             .schema(my_strct_schema)\
#             .option('header',True)\
#             .load('/Volumes/workspace/default/pysparktutorial_al_dataset_bigmartsales/BigMart Sales.csv')

In [0]:
df.printSchema()

##  Transformations

### Scenario Based questions

- Filter the data witht the content = Regular
- Slice the data with Item type = Soft Drinks and Weight < 10
- Fetch the data with Tier in (Tier 1 or Tier 2) and Outlet Size is Null

In [0]:
df.display()

In [0]:
from pyspark.sql.functions import *


### Filter the data with Fat Content = Regular

In [0]:
df.filter(col('Item_Fat_Content') == 'Regular').display()

### Slice the data with Item Type = Soft Drinks and Weight < 10

In [0]:
df.printSchema()

In [0]:
df.filter((col('Item_Type') == 'Soft Drinks') & (col("Item_Weight") < 10)).display()

### Fetch the data with Tier in (Tier 1 or Tier 2) and Outlet Size is NULL

In [0]:
df.filter((col('Outlet_Size').isNull()) & (col('Outlet_Location_Type').isin('Tier 1', 'Tier 2'))).display()

## withColumnRenamed

In [0]:
df.withColumnRenamed('Item_Weight','Item_Wt')

### withColumn

### Scenario 1: Add New column

In [0]:
df.withColumn('I_AM_Right', lit(False)).display()

In [0]:
df = df.withColumn('Flag',lit("new"))

In [0]:
df.display()

In [0]:
df.withColumn('Multiply',col('Item_Weight')*col('Item_MRP')).display()

In [0]:
df.display()

### Scenario -2: Modify Existing one

In [0]:
df = df.withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Regular","Reg"))\
    .withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Low Fat","Lf"))

df.display()

## Type Casting

In [0]:
df = df.withColumn('Item_Weight', col('Item_Weight').cast(StringType())) 

In [0]:
df.printSchema()

## DAY 3 - TRANSFORMATION INTERMEDITATE LEVEL

### SORT 

#### Sceanrio -1: Sort based on single column in ascending order

In [0]:
df.sort(col('Item_Weight').asc()).display()

#### Sceanrio -1: Sort based on single column in descending order

In [0]:
df.sort(col('Item_Weight').desc()).display()

#### Sceanrio -3: Sort by multiple columns in same order (ASC/DESC)

In [0]:
df.sort(['Item_Weight', 'Item_Visibility'], ascending = [0,0]).display()

#### Sceanrio -4: Sort by multiple column in differnt order (ASC/DESC)

In [0]:
df.sort(['Item_Weight', 'Item_Visibility'], ascending = [0,1]).display()

### Limit the select

In [0]:
df.limit(20).display()

### Drop the column

#### Sceanrio 1: Drop Single column

In [0]:
df.drop('Item_Type').display()

#### Sceanrio 2: Drop multiple columns

In [0]:
df.drop('Item_Visibility', 'Item_Type').display()

### Drop Duplicates

#### Scenario-1: Drop duplicated for all records

In [0]:
df.dropDuplicates().display()

#### Sceanrio-2: Drop Duplicates for subset of columns

In [0]:
df.drop_duplicates(subset = ['Item_Type']).display()

#### Scenario-3: Select all distinct records

In [0]:
df.distinct().display()

### Union and UnionByName

#### Prepare new DataFrame

In [0]:
data1 = [('1','kad', 23),
        ('2','sid', 36)]
schema1 = 'id STRING, name STRING, age int' 

df1 = spark.createDataFrame(data1,schema1)

data2 = [('3','rahul', 26),
        ('4','jas', 30)]
schema2 = 'id STRING, name STRING, age int' 

df2 = spark.createDataFrame(data2,schema2)

In [0]:
df1.display()

In [0]:
df2.display()

### Union

In [0]:
df1.union(df2).display()

In [0]:
data1 = [('kad', 23, '1'),
        ('sid', 46, '2')]
schema1 = 'name STRING, age int, id STRING' 

df1 = spark.createDataFrame(data1,schema1)

df1.display()

In [0]:
#df1.union(df2).display()

### UNIONBYNAME

In [0]:
df1.unionByName(df2).display()

### String Functions

- initcap
- upper
- lowert

In [0]:
df.select(initcap('Item_Type').alias('Item_Type_InitCap')).display()

In [0]:
df.select(lower('Item_Type').alias('Item_Type_Lower')).display()

In [0]:
df.select(upper('Item_Type').alias('Item_Type_Upper')).display()

### DATE FUNCTIONS

- CurrentDate()
- DATE_ADD()
- DATE_SUB()
- DATEDIFF()
- DATE_FORMAT()

In [0]:
df = df.withColumn('curr_date',current_date())

df.display()

In [0]:
df = df.withColumn('week_after',date_add('curr_date',7))

df.display()

In [0]:
df.withColumn('week_before',date_sub('curr_date',7)).display()

In [0]:
df = df.withColumn('week_before',date_add('curr_date',-7)) 

df.display()

In [0]:
df = df.withColumn('datediff',datediff('week_after','curr_date'))

df.display()

In [0]:
df = df.withColumn('week_before',date_format('week_before','dd-MM-yyyy'))

df.display()


### HANDLING NULLS

### TWO APPROACH TO HANDLE NULL RECORDS IN DATASET
- DROP NULLS
- FILLING NULLS

#### Scenario-1: Drop rows where all columns are NULL.

In [0]:
df.dropna('all').display()

#### Scenario-2: Drop rows where atleast one NULL.

In [0]:
df.dropna('any').display()

#### Scenario-3: Drop NULL records for a given subset

In [0]:
df.dropna(subset=['Outlet_Size']).display()

### Filling NULLS

#### Scenario-1: Fill all Null records with Dummy Value

In [0]:
df.fillna('NotAvailable').display()

#### Filling Nulls for a given subset

In [0]:
df.fillna('NotAvailable',subset=['Outlet_Size']).display()

### Split and Indexing

#### SPLIT THE STRING INTO LIST

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')).display()

#### SLICING OF LIST USING INDEXING

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')[1]).display()

### Explode Outlet_Type into multiple rows

In [0]:
df_exp = df.withColumn('Outlet_Type',split('Outlet_Type',' '))
df_exp.display()

In [0]:
df_exp.withColumn('Outlet_Type',explode('Outlet_Type')).display()

### array_contains()
array_contains(col, value): return Boolean column --> Checks a whether a specific literal value or column value exists within an array-type column

In [0]:
df_exp.withColumn('Type1_flag',array_contains('Outlet_Type','Type1')).display()

### Day 3: Advance PySpark Transformations

#### GroupBY
- Group the dataframe based on single column and evaluate aggregations
- Group the dataframe based on multiple columns and evaluate same aggregations
- Group the dataframe based on multiple columns and evaluate diff aggregations

In [0]:
df.display()

In [0]:
df.groupBy('Item_Type').agg(sum('Item_MRP').alias('Total_Item_MRP_BasedOnITemType')).display()

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP').alias('Total_MRP_BasedOn_ItemType_and_Size')).display()

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP').alias('Total_MRP_BasedOn_ItemType_and_Size')\
                                            ,avg('Item_MRP').alias('Avg_MRP_BasedOn_ItemType_and_Size')).display()

### collect_list
collect_list() is an aggregation function used to gather all values from a column into an array (list) within a row, preserving duplicate values. 

In [0]:
data = [('user1','book1', 'fict', 'action'),
        ('user1','book2', 'fict', 'drama'),
        ('user2','book2', 'fict', 'action'),
        ('user2','book4', 'non-fict', 'self-help'),
        ('user3','book1', 'fict', 'action')]

schema = 'user string, book string, genre string, sub_genre string'

df_book = spark.createDataFrame(data,schema)

df_book.display()

In [0]:
df_book.groupBy('genre').agg(collect_list('user')).display()

### PIVOT

In [0]:
df.groupBy('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).display()

### WHEN-OTHERWISE
Its similar to CASE STATEMENTS IN SQL
- Scenario-1: evaluate conditional column based on single ccolumn
- Scenario-2: evaluate conditional column based on multiple columns

In [0]:
# Scenario-1: evaluate conditional column based on single ccolumn
df = df.withColumn('Veg_Flag',when(col('Item_Type')=='Meat','Non-Veg').otherwise('Veg'))
df.display()

In [0]:

df.withColumn('veg_exp_flag',when(((col('veg_flag')=='Veg') & (col('Item_MRP')<100)),'Veg_Inexpensive')\
                            .when((col('veg_flag')=='Veg') & (col('Item_MRP')>100),'Veg_Expensive')\
                            .otherwise('Non_Veg')).display() 

### JOINS
- INNER JOIN
- LEFT JOIN
- RIGHT JOIN
- SELF JOIN
- CROSS JOIN
- ANTI JOIN

In [0]:
sample_data_1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

sample_schema_1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(sample_data_1,sample_schema_1)

sample_data_2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

sample_schema_2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(sample_data_2,sample_schema_2)
     


In [0]:
df1.display()

In [0]:
df2.display()

#### Inner JOIN

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'],'inner')\
    .select(df1['emp_id'], df1['emp_name'], df2['department']).display()

#### LEFT JOIN

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'],'left')\
    .select(df1['emp_id'], df1['emp_name'], df2['department']).display()

#### RIGHT JOIN

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'],'right')\
    .select(df1['emp_id'], df1['emp_name'], df2['department']).display()

#### ANTI JOIN
fetch the records from dataframe 1 that doesn't have any records in dataframe 2

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'],'anti').display()

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'],'left')\
    .filter(df2['department'].isNull()).display()

#### FULL OUTER JOIN

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'],'full').display()

### WINDOW FUNCTIONS: special functions that perform row-level calculations
- ROW_NUMBER()
- RANK()
- DENSE_RANK()

In [0]:
from pyspark.sql.window import Window

In [0]:
df.display()

In [0]:
df.withColumn('rowCol', row_number().over(Window.orderBy('Item_Weight'))).display()

In [0]:
df.withColumn('rank', rank().over(Window.orderBy(col('Item_Weight').desc())))\
    .withColumn('dense_rank', dense_rank().over(Window.orderBy(col('Item_Weight').desc())))\
        .display()

#### Scenario-1: Evaluative cummulative sum of current row along with sum of all previous rows falls under same window (partition)

In [0]:
df.withColumn('Cummlative_Sum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()


#### Scenario-2: Sum of unbounded preceding and unbounded following
Note: Usecase --> evaluate complex calcualte to get percentage and you want total sum along side

In [0]:
df.withColumn('Total_Sum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))).display()

### User Defined Functions

#### Sceanrio-1: Define UDF

In [0]:
def my_func(x):
    return x*x 

In [0]:
# Method A: Regular Python function wrapper
square_value = udf(my_func)

In [0]:
df.withColumn('MRP_SQ',square_value('Item_MRP')).display()

### DATA WRITING

In [0]:
dbutils.fs.ls("/Volumes/workspace/default/pysparktutorial_al_dataset_bigmartsales/")

#### WRITE INTO CSV

#### WRITING MODES
- append
- overwrite
- error
- ignore

In [0]:
df.write.format('csv')\
        .save('/Volumes/workspace/default/pysparktutorial_al_dataset_bigmartsales/data.csv')

#### Append

In [0]:
df.write.format('csv')\
        .mode('append')\
        .save('/Volumes/workspace/default/pysparktutorial_al_dataset_bigmartsales/data.csv')

#### Overwrite

In [0]:
df.write.format('csv')\
.mode('overwrite')\
.option('path','/Volumes/workspace/default/pysparktutorial_al_dataset_bigmartsales/data.csv')\
.save()

#### Error

In [0]:
df.write.format('csv')\
.mode('error')\
.option('path','/Volumes/workspace/default/pysparktutorial_al_dataset_bigmartsales/data.csv')\
.save()

#### Ignore

In [0]:
df.write.format('csv')\
.mode('ignore')\
.option('path','/Volumes/workspace/default/pysparktutorial_al_dataset_bigmartsales/data.csv')\
.save()

### PARQUET

In [0]:
df.write.format('parquet')\
.mode('overwrite')\
.option('path','/Volumes/workspace/default/pysparktutorial_al_dataset_bigmartsales/data.csv')\
.save()

TABLE

In [0]:
df.write.format('parquet')\
.mode('overwrite')\
.saveAsTable('new_table')